In [2]:
%pip install scikit-learn xgboost
%pip install imblearn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from imblearn.over_sampling import SMOTE
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.svm import SVC
from xgboost import XGBClassifier

import numpy as np
random_seed = 2026
np.random.seed(random_seed)

In [2]:
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

## Load Data

In [3]:
def load_data(path, channels=["delta", "theta", "alpha", "beta","gamma"]):
    # Load data
    data = pd.read_pickle(path).reset_index(drop=True)
    
    # Split data
    groups = data["subject"]
    X = data[[column for column in data.columns if any(channel in column for channel in channels)]].astype(float)
    y = data["is_mw"].astype(int)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_seed)
    
    return X, y, groups, cv

## Training
### Logistic Regression

In [ ]:
def train_logistic_regression(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed) # 4-fold for during GridSearch 

    # Train model
    lr = LogisticRegression(random_state=random_seed)
    clf = GridSearchCV(lr, parameters, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=verbose) 

    clf.fit(X,y,groups=groups)
    
    return clf

### SVM with radial basis function

In [ ]:
def train_svc(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed) # 4-fold for during GridSearch 

    # Train model
    svc = SVC(probability = True,random_state=random_seed)
    clf = GridSearchCV(svc, parameters, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=verbose) 

    clf.fit(X,y,groups=groups)
    
    return clf

### XGBoost

In [ ]:
def train_xgboost(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed) # 4-fold for during GridSearch 

    # Train model
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=random_seed, eval_metric="auc")
    clf = GridSearchCV(xgb, parameters, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=verbose) 

    clf.fit(X,y,groups=groups)
    
    return clf

### Training

In [ ]:
lr_params = {
    'C': [0.01, 0.1, 1],
    'class_weight': ["balanced"],
    'solver': ["lbfgs"],
    'max_iter': [1000],
    'warm_start': [True, False]
}

svm_params = {
    'C': [0.01,0.1,1],
    'gamma': ["auto", "scale"]
}

xgb_params = {
    'learning_rate': [0.01],
    'n_estimators': [200,300],
    'max_depth': [10,15],
    'reg_alpha': [1,3],
}

In [8]:
path = "data/preprocessed/data_channels_2000ms.pkl"
X, y, groups, cv = load_data(path) # This loads all columns. You can filter for specific columns by adding filter substrings to the 'channels' parameter list.
X.head()

,delta_Fp1,delta_AF7,delta_AF3,delta_F1,delta_F3,delta_F5,delta_F7,delta_FT7,delta_FC5,delta_FC3,...,gamma_CP4,gamma_CP2,gamma_P2,gamma_P4,gamma_P6,gamma_P8,gamma_P10,gamma_PO8,gamma_PO4,gamma_O2
0,2.462158,-0.786966,1.453841,-0.173519,-0.769492,-0.287671,-0.016817,0.106244,-0.263572,-0.469618,...,1.657072,0.698362,0.887055,1.705181,1.210251,7.597670,6.660648,4.159779,1.046496,0.598613
1,-0.763733,-0.763764,-0.839963,-0.751324,-0.641978,-0.288063,-0.079681,-0.898491,-0.984690,-0.238345,...,0.712813,1.334872,0.824546,1.228759,0.946114,4.261911,2.271643,1.681627,0.584873,-0.070074
2,1.837950,0.279270,2.548398,3.484596,3.426255,0.389319,-0.071008,-0.581717,-0.112547,1.432430,...,2.790258,1.954135,0.441275,1.204265,0.282846,0.065358,0.035225,-0.480892,-0.210074,-0.642990
3,1.021164,-0.041133,0.208394,0.182622,0.679224,0.241317,0.008427,-0.222973,-0.396165,0.456203,...,0.522042,0.611324,0.837795,0.429085,0.173270,1.052851,0.386238,0.139689,0.180052,-0.365811
4,0.363276,-0.464267,-0.007265,0.148546,0.289437,-0.376768,-0.103516,-0.918463,-0.986162,-0.488684,...,0.236916,0.645471,0.714078,0.989184,-0.021414,1.342126,0.248375,0.056425,0.155765,-0.487687


In [ ]:
verbose = 2

# Save performance metrics to report them later
logreg_aucs = []
logreg_best_params = None
svc_aucs = []
svc_best_params = None
xgb_aucs = []
xgb_best_params = None

for fold, (train_idxs, test_idxs) in enumerate(cv.split(X, y, groups)):
    # Train test split for present split
    X_trainval, y_trainval = X.iloc[train_idxs].reset_index(drop=True), y[train_idxs].reset_index(drop=True)
    X_test, y_test = X.iloc[test_idxs].reset_index(drop=True), y[test_idxs].reset_index(drop=True)

    # Oversample underrepresented class
    smote = SMOTE(sampling_strategy='minority', random_state=random_seed)
    X_trainval_sm, y_trainval_sm = smote.fit_resample(X_trainval, y_trainval)

    # Update groups
    groups_train = groups[train_idxs]
    groups_test = groups[test_idxs]
    rest = len(X_trainval_sm) - len(X_trainval)
    groups_train = pd.concat([groups_train, pd.Series(np.random.choice(groups_train.unique(), size = rest))]) # This fills up the groups variable with random subject ids. This is okay because they are synthetic; they just need to have a value for the code to work

    print(f"Fold {fold}")

    # Logistic Regression
    best_logreg_clf = train_logistic_regression(X_trainval_sm, y_trainval_sm, groups_train, parameters=lr_params) # GridSearch
    perf_auc = roc_auc_score(y_test, best_logreg_clf.predict_proba(X_test)[:,1]) # Evaluation
    logreg_aucs.append(perf_auc)
    if perf_auc == max(logreg_aucs):
        logreg_best_params = best_logreg_clf.best_params_
    perf_f1 = f1_score(y_test, best_logreg_clf.predict(X_test))
    print(f"LogisticRegression({best_logreg_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")

    # Support Vector Classifier
    best_svm_clf = train_svc(X_trainval_sm, y_trainval_sm, groups_train, parameters=svm_params)
    perf_auc = roc_auc_score(y_test, best_svm_clf.decision_function(X_test))
    svc_aucs.append(perf_auc)
    if perf_auc == max(svc_aucs):
        svc_best_params = best_svm_clf.best_params_
    perf_f1 = f1_score(y_test, best_svm_clf.predict(X_test))
    print(f"SVC({best_svm_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")

    # XGBoost Classifier
    best_xgb_clf = train_xgboost(X_trainval_sm, y_trainval_sm, groups_train, parameters=xgb_params)
    perf_auc = roc_auc_score(y_test, best_xgb_clf.predict_proba(X_test)[:,1])
    xgb_aucs.append(perf_auc)
    if perf_auc == max(xgb_aucs):
        xgb_best_params = best_xgb_clf.best_params_
    perf_f1 = f1_score(y_test, best_xgb_clf.predict(X_test))
    print(f"XGBoost({best_xgb_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")
    
    print()


Fold 0
Fitting 4 folds for each of 8 candidates, totalling 32 fits


In [ ]:
print("--- All ---")
print(f"Mean Logistic Regression  AUC = {sum(logreg_aucs)/len(logreg_aucs)} ({logreg_best_params})") 
print(f"Mean SVC AUC = {sum(svc_aucs)/len(svc_aucs)} ({svc_best_params})")
print(f"Mean XGBoost AUC = {sum(xgb_aucs)/len(xgb_aucs)} ({xgb_best_params})")

--- All ---
Mean Logistic Regression  AUC = 0.5546344190655784 ({'C': 1, 'class_weight': 'balanced', 'max_iter': 1000, 'solver': 'lbfgs'})
Mean SVC AUC = 0.5337811779654602 ({'C': 1, 'gamma': 'auto', 'kernel': 'poly'})
Mean XGBoost AUC = 0.50696939385821 ({'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_alpha': 1})


### Get Feature Importances

In [16]:
path = "data/preprocessed/data_channels_2000ms.pkl"

In [33]:
# Retrain "All" model
X, y, groups, cv = load_data(path)
best_params_all = {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_alpha': 1}
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
xgb_all = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=random_seed, eval_metric="auc", **best_params_all)
xgb_all.fit(X,y)

# Print feature importances per channel (sum, mean, std)
print("--- Feature Importances per Channel ---")
for band in ["alpha", "beta","gamma","delta", "theta"]:
    band_specific_feature_importances = xgb_all.feature_importances_[[band in column for column in X.columns]]
    print(f"{band} - {band_specific_feature_importances.sum()} (Mean = {band_specific_feature_importances.mean()}; SD = {band_specific_feature_importances.std()})")
    display()

--- Feature Importances per Channel ---
alpha - 0.19574137032032013 (Mean = 0.003058458911255002; SD = 0.00087429687846452)
beta - 0.2192164808511734 (Mean = 0.0034252575132995844; SD = 0.0011904011480510235)
gamma - 0.22026047110557556 (Mean = 0.003441569861024618; SD = 0.0008349965792149305)
delta - 0.17510002851486206 (Mean = 0.0027359379455447197; SD = 0.000926378823351115)
theta - 0.18968158960342407 (Mean = 0.002963774837553501; SD = 0.0014819989446550608)


In [ ]:
# Retrain "Theta"
X, y, groups, cv = load_data(path, channels=["theta"])
best_params_theta = {'learning_rate': 0.01, 'max_depth': 15, 'n_estimators': 300, 'reg_alpha': 1}
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
xgb_theta = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=random_seed, eval_metric="auc", **best_params_theta)
xgb_theta.fit(X,y)
xgb_theta.feature_importances_

array([0.01341207, 0.01347036, 0.01442302, 0.0128604 , 0.01594581,
       0.01310224, 0.01563223, 0.0150395 , 0.0134599 , 0.01407742,
       0.01539253, 0.01385969, 0.01320453, 0.01219457, 0.01515923,
       0.0156785 , 0.01543811, 0.01389861, 0.01520607, 0.01337508,
       0.01338665, 0.01316003, 0.01402587, 0.01828864, 0.02271202,
       0.01764891, 0.01534392, 0.01264799, 0.0180807 , 0.01576025,
       0.0166458 , 0.01657051, 0.01314338, 0.01581788, 0.01623123,
       0.0165428 , 0.01574782, 0.01456846, 0.01719348, 0.01462904,
       0.01366888, 0.01652761, 0.01600782, 0.0117954 , 0.01561437,
       0.01723124, 0.0188775 , 0.01308779, 0.01487084, 0.01336633,
       0.01764703, 0.01686746, 0.01444182, 0.01655602, 0.0145658 ,
       0.01673548, 0.01787024, 0.01567422, 0.0160271 , 0.01511257,
       0.02101227, 0.01658268, 0.03120256, 0.01568176], dtype=float32)

In [ ]:
# Retrain "Alpha"
X, y, groups, cv = load_data(path, channels="alpha")
best_params_alpha = {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_alpha': 1}
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
xgb_alpha = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=random_seed, eval_metric="auc", **best_params_alpha)
xgb_alpha.fit(X,y)
xgb_alpha.feature_importances_

array([0.00245892, 0.00300389, 0.00229649, 0.00308352, 0.00194158,
       0.00239303, 0.00136712, 0.00160504, 0.0023745 , 0.00270718,
       0.00246657, 0.00208332, 0.00255656, 0.0045543 , 0.00535977,
       0.00389881, 0.00312125, 0.00278973, 0.00323298, 0.00338775,
       0.00267987, 0.00257333, 0.00273739, 0.00294143, 0.00411374,
       0.00384341, 0.00384497, 0.00192796, 0.0043191 , 0.00160407,
       0.00320596, 0.00197821, 0.00341894, 0.00247194, 0.00332932,
       0.00375984, 0.00269763, 0.00189259, 0.00193413, 0.00304454,
       0.00189888, 0.00496785, 0.00343631, 0.00315512, 0.00335014,
       0.00231555, 0.00204979, 0.00197628, 0.00137471, 0.00221322,
       0.00135173, 0.00462319, 0.0032323 , 0.00269558, 0.00103533,
       0.00278812, 0.00276765, 0.00345589, 0.0029634 , 0.00275984,
       0.00429417, 0.00298746, 0.00157905, 0.0037424 , 0.0014014 ,
       0.0032913 , 0.00169234, 0.00149354, 0.00699164, 0.00201716,
       0.00304358, 0.00357977, 0.00219396, 0.0018127 , 0.00232